In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 49.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


## Local Inference on GPU
Model page: https://huggingface.co/dbmdz/bert-base-turkish-cased

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/dbmdz/bert-base-turkish-cased)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
# Load model directly
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")
model = AutoModel.from_pretrained("dbmdz/bert-base-turkish-cased")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import pandas as pd

df = pd.read_csv("datasets/anlamver-final.csv", encoding='windows-1254', sep=';')
df.head()

,QID,W1,W2,Sim,Rel,S,D,R,U,SR,...,EstMer,W1-RWG,W2-RWG,RWMin,W1-DG,W2-DG,DGMax,W1-IG,W2-IG,IGMax
0,1,mızrap,barınak,"0,166666667","0,083333333",False,True,False,True,False,...,0,2,3,2,0,1,1,0,0,0
1,2,kırmızı,gül,"1,166666667","7,166666667",False,True,True,False,False,...,0,5,5,5,0,0,0,0,0,0
2,3,suçlu,şüphe,"2,416666667",7,False,True,True,False,False,...,0,4,4,4,1,0,1,0,0,0
3,4,laikçiler,sekülerizmciler,9,"9,416666667",True,False,True,False,True,...,0,2,0,0,2,0,2,0,0,0
4,5,bitki,zeytin,"3,666666667","6,75",False,True,True,False,False,...,0,4,4,4,0,0,0,0,0,0


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

def count_subtokens(text):
    return len(tokenizer.tokenize(str(text)))

df["word1_subtokens"] = df["W1"].apply(count_subtokens)
df["word2_subtokens"] = df["W2"].apply(count_subtokens)

df.head()

,QID,W1,W2,Sim,Rel,S,D,R,U,SR,...,W2-RWG,RWMin,W1-DG,W2-DG,DGMax,W1-IG,W2-IG,IGMax,word1_subtokens,word2_subtokens
0,1,mızrap,barınak,"0,166666667","0,083333333",False,True,False,True,False,...,3,2,0,1,1,0,0,0,2,2
1,2,kırmızı,gül,"1,166666667","7,166666667",False,True,True,False,False,...,5,5,0,0,0,0,0,0,1,1
2,3,suçlu,şüphe,"2,416666667",7,False,True,True,False,False,...,4,4,1,0,1,0,0,0,1,1
3,4,laikçiler,sekülerizmciler,9,"9,416666667",True,False,True,False,True,...,0,0,2,0,2,0,0,0,2,4
4,5,bitki,zeytin,"3,666666667","6,75",False,True,True,False,False,...,4,4,0,0,0,0,0,0,1,1


In [7]:
df["word1_fragmented"] = df["word1_subtokens"] > 1
df["word2_fragmented"] = df["word2_subtokens"] > 1

In [10]:
#4. Statistics for thesiss
print("Word1 averaged subtoken:", df["word1_subtokens"].mean())
print("Word2 averaged subtoken:", df["word2_subtokens"].mean())

print("Fragmented word1 ratio:", df["word1_fragmented"].mean())
print("Fragmented word2 ratio:", df["word2_fragmented"].mean())

Word1 averaged subtoken: 1.528
Word2 averaged subtoken: 1.564
Fragmented word1 ratio: 0.374
Fragmented word2 ratio: 0.386


In [11]:
# most fragmented
df_sorted = df.sort_values(by="word1_subtokens", ascending=False)

df_sorted[["W1", "word1_subtokens"]].head(10)

,W1,word1_subtokens
438,anarşist,4
382,primatçaları,4
389,primatçaları,4
98,kavramsallaştırsın,4
223,gençmişçesine,4
197,saldırmaktaydılar,4
205,biçimlendirilmişlerdir,4
36,primatçaları,4
227,kavramsallaştırsın,4
287,pimpirikli,4


In [12]:
def get_subtokens(text):
    return tokenizer.tokenize(str(text))

df["word1_tokens"] = df["W1"].apply(get_subtokens)
df["word2_tokens"] = df["W2"].apply(get_subtokens)

df.head()

,QID,W1,W2,Sim,Rel,S,D,R,U,SR,...,DGMax,W1-IG,W2-IG,IGMax,word1_subtokens,word2_subtokens,word1_fragmented,word2_fragmented,word1_tokens,word2_tokens
0,1,mızrap,barınak,"0,166666667","0,083333333",False,True,False,True,False,...,1,0,0,0,2,2,True,True,"[mız, ##rap]","[barın, ##ak]"
1,2,kırmızı,gül,"1,166666667","7,166666667",False,True,True,False,False,...,0,0,0,0,1,1,False,False,[kırmızı],[gül]
2,3,suçlu,şüphe,"2,416666667",7,False,True,True,False,False,...,1,0,0,0,1,1,False,False,[suçlu],[şüphe]
3,4,laikçiler,sekülerizmciler,9,"9,416666667",True,False,True,False,True,...,2,0,0,0,2,4,True,True,"[laik, ##çiler]","[sek, ##üler, ##izm, ##ciler]"
4,5,bitki,zeytin,"3,666666667","6,75",False,True,True,False,False,...,0,0,0,0,1,1,False,False,[bitki],[zeytin]


In [13]:
df.to_csv("anlamver_with_tokens.csv", index=False)

In [14]:
df["total_subtokens"] = df["word1_subtokens"] + df["word2_subtokens"]

In [15]:
df['Sim'] = df['Sim'].str.replace(',', '.', regex=False).astype(float)
df[['total_subtokens', 'Sim']].corr(method='spearman')

,total_subtokens,Sim
total_subtokens,1.000000,-0.243753
Sim,-0.243753,1.000000


In [17]:
import pandas as pd

df = pd.read_csv("/content/anlamver_with_tokens.csv")

# Temiz çift (ikisi de 1 token)
df["clean_pair"] = (df["word1_subtokens"] == 1) & (df["word2_subtokens"] == 1)

# Bölünmüş çift (en az biri 2+ token)
df["split_pair"] = (df["word1_subtokens"] > 1) | (df["word2_subtokens"] > 1)

# Sayılar
clean_count = df["clean_pair"].sum()
split_count = df["split_pair"].sum()
total = len(df)

print("Toplam çift:", total)
print("Temiz çift (1-1):", clean_count)
print("Bölünmüş çift (en az biri 2+):", split_count)

# Yüzde
print("\nYüzde olarak:")
print("Temiz çift %:", clean_count / total * 100)
print("Bölünmüş çift %:", split_count / total * 100)

Toplam çift: 500
Temiz çift (1-1): 225
Bölünmüş çift (en az biri 2+): 275

Yüzde olarak:
Temiz çift %: 45.0
Bölünmüş çift %: 55.00000000000001
